In [1]:
import pandas as pd
import pyreadr
import geopandas as gpd
import glob
import iris
import iris.coord_categorisation
# # import random
# import functions
# importlib.reload(functions)
# from functions import *

data_dir = "/scratch/hydro4/users/kv25483/FDRI/Rain_gauge_optimisation/Data/"

In [2]:
# Load rain gauge metadata
rain_meta_cmd = pd.read_csv(data_dir + "rain_meta_for_CMD.csv")

# Convert to GeoDataFrame (BNG coordinates)
rain_meta_sf = gpd.GeoDataFrame(rain_meta_cmd, geometry=gpd.points_from_xy(rain_meta_cmd["x_2"], rain_meta_cmd["y_2"]),
                                crs="EPSG:27700")
# rain_meta_sf['station_name'].tolist()

### Get AWS rain gauge data for the Carreg and Tan gauges

In [3]:
rain_gauge = pd.read_csv(data_dir  + "RainGaugeData/Plynlimon/aws_hourly_qc.csv")
lookup = pd.read_csv(data_dir  + "RainGaugeData/Plynlimon/aws_site_metadata.csv")

carreg = rain_gauge[rain_gauge['id'].isin(['CW', 'CW2', 'CW3'])].copy()
tan = rain_gauge[rain_gauge['id'].isin(['TN', 'TN2', 'TN3', 'TW', 'TM'])].copy()

### Get NRW rain gauge data for the Dolydd and Nantgwynmain gauges

In [4]:
nrw_gauges = pyreadr.read_r(data_dir  + "RainGaugeData/NRW/NRW_rainfall_all.rds") # also works for RData
nrw_gauges = nrw_gauges[None] # extract the pandas data frame 

dolydd = nrw_gauges[nrw_gauges['station'].isin(['DOLYDD'])].copy()
nantgwynmain = nrw_gauges[nrw_gauges['station'].isin(['NANTGWYNMAIN'])].copy()

### Convert to annual values

In [5]:
def extract_annual_values (df, date_column, precip_column, source):
    # 1. Ensure datetime
    if source == 'NRW':
        df[date_column] = pd.to_datetime(df[date_column],dayfirst=True, format='mixed')
    elif source == 'ARW':
        df[date_column] = pd.to_datetime(df[date_column])
        
    # 2. Extract year
    df['year'] = df[date_column].dt.year

    # 3. Annual totals
    annual_rainfall = df.groupby('year')[precip_column].sum()

    # 4. Mean annual rainfall
    mean_annual_rainfall = annual_rainfall.mean()

    return annual_rainfall

carreg_annual_means = extract_annual_values(carreg, 'date_time', 'precipitation', 'ARW')
tan_annual_means = extract_annual_values(tan, 'date_time', 'precipitation', 'ARW')
nantgwynmain_annual_means = extract_annual_values(nantgwynmain, 'Time stamp', 'rain_mm', 'NRW')
dolydd_annual_means = extract_annual_values(dolydd, 'Time stamp', 'rain_mm', 'NRW')

### Get equivalent rain gauge data

In [6]:
ceh_gear_dir = "/scratch/hydro4/shared_data/climate_observed/CEH-GEAR/hourly/1km/"

### Get CEH-GEAR data from 1990 to 2016

In [7]:
filenames=[]

for year in range(1990,2017):
    general_filename = ceh_gear_dir + f'CEH-GEAR-1hr-v2_{year}*'
    for filename in glob.glob(general_filename):
        filenames.append(filename)

monthly_cubes_list = iris.load(filenames)

rain_cubes = [c for c in monthly_cubes_list if c.name() == 'rainfall_amount']
rain_cubes = iris.cube.CubeList(rain_cubes)
obs_cube = rain_cubes[0]

cubes = iris.load(filenames)

rain_cubes = iris.cube.CubeList([c for c in cubes if c.name() == 'rainfall_amount'])

for c in rain_cubes:
    c.attributes = {}   # 🔥 remove all conflicting attributes

ceh_gear = rain_cubes.concatenate_cube()

/home/kv25483/anaconda3/envs/geo_env2/lib/python3.10/site-packages/iris/__init__.py:680: FutureWarning: Ignoring a datum in netCDF load for consistency with existing behaviour. In a future version of Iris, this datum will be applied. To apply the datum when loading, use the iris.FUTURE.datum_support flag.
  cubes = _load_collection(uris, constraints, callback).combined().cubes()


### Find the annual mean

In [8]:
iris.coord_categorisation.add_season_year(ceh_gear, 'time', name='year', iris.FUTURE.date_microseconds = True)
annual_mean = ceh_gear.aggregated_by(["year"], iris.analysis.MEAN)

SyntaxError: expression cannot contain assignment, perhaps you meant "=="? (2120553703.py, line 1)

### Plot the annual mean from Gridded data for each gauge station

In [ ]:
DOLYDD = rain_meta_sf[rain_meta_sf['station_name']=='DOLYDD']
x = float(DOLYDD['x'][0])
y = float(DOLYDD['y'][0])

dolydd_annual_mean_gridded = annual_mean.interpolate([('projection_x_coordinate', x),('projection_y_coordinate', y)],
                                             iris.analysis.Nearest())

dolydd_annual_mean_gridded_data = dolydd_annual_mean_gridded.data  # should now be 1D (n_years,)
years = dolydd_annual_mean_gridded.coord('year').points

plt.scatter(years, dolydd_annual_mean_gridded_data)

In [ ]:
dolydd_annual_means